In [2]:
import tensorflow as tf
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

In [20]:
train_normal = []
train_pneumonia = []
test_normal = []
test_pneumonia = []
etiquetas = []

IMG_SIZE = (224, 224)  # Tamaño para todas las imágenes
rutas_pneumonia = tf.data.Dataset.list_files('./imagenes/test/PNEUMONIA/*.jpeg')
rutas_normal = tf.data.Dataset.list_files('./imagenes/test/NORMAL/*.jpeg')

# TODO Simplificar esto con una funcion y un map (como en el ejercicio 2)
for imgRoute in rutas_normal:
    img = tf.io.read_file(imgRoute)
    img_tensor = tf.image.decode_jpeg(img, channels = 3)
    img_resized = tf.image.resize(img_tensor, IMG_SIZE)  # Redimensionar a tamaño consistente
    # img_flattened = img_resized.flatten()  # Aplanar a 1D
    train_normal.append(img_resized)
    # Llenamos las etiquetas de 0 (normal) por cada foto de rayos-x sin pneumonia
    etiquetas.append(0)

train_normal_tensor = tf.convert_to_tensor(train_normal)
train_normal_df = tf.data.Dataset(train_normal_tensor)

for imgRoute in rutas_pneumonia:
    img = tf.io.read_file(imgRoute)
    img_tensor = tf.image.decode_jpeg(img, channels = 3)
    img_resized = tf.image.resize(img_tensor, IMG_SIZE)  # Redimensionar a tamaño consistente
    # img_flattened = img_resized.flatten()  # Aplanar a 1D
    train_pneumonia.append(img_resized)
    # Llenamos las etiquetas de 1 (pneumonia) por cada foto de rayos-x con pneumonia
    etiquetas.append(1)

train_normal_tensor = tf.convert_to_tensor(train_normal, dtype=tf.float32)
train_pneumonia_tensor = tf.convert_to_tensor(train_pneumonia, dtype=tf.float32)

train_normal_df = tf.data.Dataset.from_tensor_slices(train_normal_tensor)
train_pneumonia_df = tf.data.Dataset.from_tensor_slices(train_pneumonia_tensor)
etiquetas_df = tf.data.Dataset.from_tensor_slices(etiquetas)

dataset_imagenes = train_normal_df.concatenate(train_pneumonia_df)

dataset_final = tf.data.Dataset.zip((dataset_imagenes, etiquetas_df))

scaler = StandardScaler()
# X_normalizado = scaler.fit_transform(imagenes_train)
print(dataset_final.element_spec)
print('Total imágenes:', len(train_normal) + len(train_pneumonia))

TypeError: Can't instantiate abstract class DatasetV2 with abstract methods _inputs, element_spec

In [11]:
modelo = tf.keras.Sequential([
        tf.keras.layers.Dense(32, activation='relu', input_shape=(10,)),
        tf.keras.layers.Dense(16, activation='relu'),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])
modelo.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 32)             │           352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 897 (3.50 KB)

 Trainable params: 897 (3.50 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Config
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

# 1) Crear dataset con rutas + etiqueta
ds_normal = tf.data.Dataset.list_files('./imagenes/test/NORMAL/*.jpeg')
ds_normal = ds_normal.map(lambda p: (p, 0))

ds_pneumonia = tf.data.Dataset.list_files('./imagenes/test/PNEUMONIA/*.jpeg')
ds_pneumonia = ds_pneumonia.map(lambda p: (p, 1))

dataset = ds_normal.concatenate(ds_pneumonia)

# 2) Cargar y preparar imagen
def preprocess(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32) / 255.0
    return img, label

dataset = dataset.map(preprocess)

# 3) shuffle + batch + prefetch
dataset = dataset.shuffle(1000).batch(BATCH_SIZE).prefetch(AUTOTUNE)

print(dataset.element_spec)

model = tf.keras.Sequential([
        tf.keras.layers.Dense(32, activation='relu', input_shape=(10,)),
        tf.keras.layers.Dense(16, activation='relu'),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(224, 224, 3)),
    tf.keras.layers.Conv2D(32, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(64, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(128, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

model.summary()

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history = model.fit(dataset, epochs=10)
resultados = model.evaluate(dataset, return_dict=True)
print(resultados)